# ProofAgent™ — Log-based evaluation (Google Colab)

Send a full **conversation** as `logs` (each row: `turn_index`, `user_message`, `agent_answer`). The backend evaluates in one pipeline—no interactive question loop.

| Term | Meaning |
|------|--------|
| **Log-based** | Single `start_run(logs=[...])`. |
| **Polling** | Status moves to `completed`; use **verbose polling** to see elapsed time and turn progress. |
| **Report** | Scores + **transcript** with per-turn details. |

Install the SDK like in `colab_judge_led.ipynb` (same `GIT_URL` / upload pattern).


In [ ]:
import subprocess, sys, os
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])
GIT_URL = os.environ.get("PROOFAGENT_SDK_GIT", "")
SDK_ROOT = os.environ.get("PROOFAGENT_SDK_ROOT", "/content/Proofagent-python-sdk")
if GIT_URL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", GIT_URL])
elif os.path.isfile(os.path.join(SDK_ROOT, "pyproject.toml")):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", SDK_ROOT])
print("done")


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["PROOFAGENT_API_KEY"] = userdata.get("PROOFAGENT_API_KEY")
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
except ImportError:
    pass
assert os.environ.get("PROOFAGENT_API_KEY", "").strip()


In [ ]:
import asyncio
import os
import sys
from pathlib import Path
from typing import Any

from proofagent import ProofAgentClient

for r in [Path.cwd(), Path("/content/Proofagent-python-sdk")]:
    if (r / "examples" / "report_helpers.py").is_file():
        sys.path.insert(0, str(r / "examples"))
        break
from report_helpers import poll_until_complete_verbose, print_full_evaluation_report

logs: list[dict[str, Any]] = [
    {"turn_index": 1, "user_message": "I was charged twice", "agent_answer": "Let me check invoices."},
    {"turn_index": 2, "user_message": "Refund?", "agent_answer": "I can start a refund."},
]

async def main():
    async with ProofAgentClient.from_env() as client:
        kw: dict = {"logs": logs, "agent_role": "Billing assistant"}
        if os.environ.get("OPENAI_API_KEY", "").strip():
            kw.update(llm_api_key=os.environ["OPENAI_API_KEY"], llm_provider="openai", llm_model="gpt-4o-mini")
        run = await client.start_run(**kw)
        rid = run["data"]["run_id"]
        print("Run id:", rid)
        await poll_until_complete_verbose(client, rid, timeout_seconds=900.0, poll_every_seconds=3.0)
        rep = await client.get_report(rid)
        print_full_evaluation_report(rep)

await main()
